# E-Commerce Analytics System
# Notebook 4: SQL Analytics (Basic & Intermediate)

This notebook connects to the SQLite database and executes the required analytical SQL queries.


In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

conn=sqlite3.connect(Path("database")/"ecommerce.db")

def run_query(title,query):
    print("="*80)
    print(title)
    df=pd.read_sql(query,conn)
    display(df.head(20))
    return df


## Revenue per Category

In [ ]:
run_query("""Revenue per Category""", """SELECT p.category,
ROUND(SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)),2) AS total_revenue
FROM order_items oi
JOIN products p ON oi.product_id=p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;""")

## Top 10 Customers

In [ ]:
run_query("""Top 10 Customers""", """SELECT c.customer_id,c.customer_name,
ROUND(SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)),2) total_order_value
FROM customers c
JOIN orders o ON c.customer_id=o.customer_id
JOIN order_items oi ON o.order_id=oi.order_id
GROUP BY c.customer_id,c.customer_name
ORDER BY total_order_value DESC
LIMIT 10;""")

## Monthly Order Count

In [ ]:
run_query("""Monthly Order Count""", """SELECT strftime('%Y-%m',order_date) month,
COUNT(*) order_count
FROM orders
GROUP BY month
ORDER BY month DESC
LIMIT 12;""")

## Revenue per Customer

In [ ]:
run_query("""Revenue per Customer""", """SELECT c.customer_name,
ROUND(SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)),2) revenue
FROM customers c
JOIN orders o ON c.customer_id=o.customer_id
JOIN order_items oi ON o.order_id=oi.order_id
GROUP BY c.customer_name
ORDER BY revenue DESC;""")

## Top Products by Quantity

In [ ]:
run_query("""Top Products by Quantity""", """SELECT p.product_name,
SUM(oi.quantity) quantity_sold
FROM products p
JOIN order_items oi ON p.product_id=oi.product_id
GROUP BY p.product_name
ORDER BY quantity_sold DESC
LIMIT 10;""")

## Top Products by Revenue

In [ ]:
run_query("""Top Products by Revenue""", """SELECT p.product_name,
ROUND(SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)),2) revenue
FROM products p
JOIN order_items oi ON p.product_id=oi.product_id
GROUP BY p.product_name
ORDER BY revenue DESC
LIMIT 10;""")

## Average Order Value by Segment

In [ ]:
run_query("""Average Order Value by Segment""", """SELECT c.customer_type,
ROUND(AVG(order_total),2) avg_order_value
FROM(
SELECT o.order_id,o.customer_id,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) order_total
FROM orders o
JOIN order_items oi ON o.order_id=oi.order_id
GROUP BY o.order_id,o.customer_id
)t
JOIN customers c ON t.customer_id=c.customer_id
GROUP BY c.customer_type;""")

## Customers With No Delivered Orders

In [ ]:
run_query("""Customers With No Delivered Orders""", """SELECT DISTINCT c.customer_id,c.customer_name
FROM customers c
JOIN orders o ON c.customer_id=o.customer_id
WHERE c.customer_id NOT IN(
SELECT customer_id FROM orders WHERE status='DELIVERED'
);""")

## Products Having More Returns Than Purchases

In [ ]:
run_query("""Products Having More Returns Than Purchases""", """SELECT p.product_name,
SUM(CASE WHEN oi.quantity<0 THEN 1 ELSE 0 END) returns,
SUM(CASE WHEN oi.quantity>0 THEN 1 ELSE 0 END) purchases
FROM products p
JOIN order_items oi ON p.product_id=oi.product_id
GROUP BY p.product_name
HAVING returns>purchases;""")

## Return Rate by Category

In [ ]:
run_query("""Return Rate by Category""", """SELECT p.category,
ROUND(100.0*SUM(CASE WHEN oi.quantity<0 THEN 1 ELSE 0 END)/COUNT(*),2) return_rate
FROM products p
JOIN order_items oi ON p.product_id=oi.product_id
GROUP BY p.category;""")

## Close Connection

In [ ]:
conn.close()
print("Analysis completed successfully.")